## Imports

In [87]:
import os
import duckdb
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata

pd.options.display.float_format = '{:,.2f}'.format

%alias gcloud gcloud


## Provide your google credentials

In [2]:
auth.authenticate_user()
print('Authenticated')

Authenticated


## Authenticate to Hugging Face

In [5]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account


'rwcitek'

In [6]:
hf_repo = "BigQuery"
os.environ["HF_REPO"] = hf_repo
hf_repo


'BigQuery'

In [7]:
# Retrieve the token from Colab's secret manager
_ = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"

'hf_ ... dQPb'

In [8]:
# Log in automatically without an interactive prompt
! hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [10]:
gcloud projects list --sort-by=projectId

PROJECT_ID                   NAME                    PROJECT_NUMBER  ENVIRONMENT
cool-monolith-286222         Data Science            271608828771
data-science-project-321016  Data Science Project    540766130804
gen-lang-client-0664606651   Default Gemini Project  566992509144


To take a quick anonymous survey, run:
  $ gcloud survey



## Declare the Cloud billing project ID

In [11]:
billing_project_id = userdata.get('bq_billing_project_id')


In [12]:
# Create client object
client = bigquery.Client(project=billing_project_id)


## Get Dataset metadata

### How many data sets are in the bigquery-public-data project?


In [13]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.
dataset_list = [dataset.dataset_id for dataset in datasets]


In [14]:
len(dataset_list)


382

In [15]:
dataset_list[:10]


['agent_analytic_sample_dataset',
 'america_health_rankings',
 'austin_311',
 'austin_bikeshare',
 'austin_crime',
 'austin_incidents',
 'austin_waste',
 'baseball',
 'bbc_news',
 'biglake-public-nyc-taxi-iceberg.public_data']

### For each data set collect metadata.

In [16]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

dataset_metadata = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
    # if i >= 3: break

    fdi = f"{dataset.project}.{dataset.dataset_id}"
    dataset_info = client.get_dataset(fdi)
    print(f"{i:03}: {dataset_info.dataset_id}")

    # Append relevant metadata to the list
    dataset_metadata.append({
        "Dataset ID": dataset_info.dataset_id,
        "Description": dataset_info.description,
        "Location": dataset_info.location,
        "Created Time": dataset_info.created,
        "Last Modified Time": dataset_info.modified,
        "Labels": dataset_info.labels,
        "Access Entries": dataset_info.access_entries,
    })

# Create a DataFrame from the list of data
dataset_metadata_df = pd.DataFrame(dataset_metadata)
dataset_metadata_df[:10]

000: agent_analytic_sample_dataset
001: america_health_rankings
002: austin_311
003: austin_bikeshare
004: austin_crime
005: austin_incidents
006: austin_waste
007: baseball
008: bbc_news
009: biglake-public-nyc-taxi-iceberg.public_data
010: bigqueryml_ncaa
011: bitcoin_blockchain
012: blackhole_database
013: blockchain_analytics_ethereum_mainnet_us
014: bls
015: bls_qcew
016: breathe
017: broadstreet_adi
018: catalonian_mobile_coverage
019: catalonian_mobile_coverage_eu
020: census_bureau_acs
021: census_bureau_construction
022: census_bureau_international
023: census_bureau_usa
024: census_opportunity_atlas
025: census_utility
026: cfpb_complaints
027: chicago_crime
028: chicago_taxi_trips
029: clemson_dice
030: cloud_storage_geo_index
031: cms_codes
032: cms_medicare
033: cms_synthetic_patient_data_omop
034: country_codes
035: covid19_aha
036: covid19_covidtracking
037: covid19_ecdc
038: covid19_ecdc_eu
039: covid19_genome_sequence
040: covid19_geotab_mobility_impact
041: covid19_ge

,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels,Access Entries
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
1,america_health_rankings,America Health Rankings,US,2021-12-30 19:32:54.441000+00:00,2025-07-22 10:45:22.624000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
2,austin_311,,US,2017-05-11 18:44:42.673000+00:00,2025-07-22 10:43:48.820000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
3,austin_bikeshare,Austin Bikeshare dataset,US,2017-05-16 19:25:37.054000+00:00,2025-07-22 10:44:24.011000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
4,austin_crime,Austin Crime dataset,US,2017-05-11 18:29:25.469000+00:00,2025-07-22 10:45:15.608000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
5,austin_incidents,,US,2017-05-16 19:25:51.354000+00:00,2025-07-22 10:45:12.508000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
6,austin_waste,austin waste and diversion,US,2017-05-16 19:25:20.941000+00:00,2025-07-22 10:45:18.110000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
7,baseball,Data Updated On : 2016-10-24\nOverview: This p...,US,2016-10-18 17:08:24.118000+00:00,2025-07-23 09:17:37.113000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, specialGroup=proje..."
8,bbc_news,Data Updated On : 2017-06-07\n,US,2017-06-06 15:21:18.540000+00:00,2026-10-13 07:12:32.569000+00:00,"{'dg_data_source': 'public', 'dg_data_context'...","[<AccessEntry: role=WRITER, groupByEmail=bigqu..."
9,biglake-public-nyc-taxi-iceberg.public_data,None,us-central1,2025-11-14 17:50:35.493000+00:00,2025-11-14 17:50:35.493000+00:00,{'iceberg_rest_catalog': 'biglake-public-nyc-t...,[]


In [17]:
dataset_metadata_df.drop(columns=["Access Entries"])[:3]


,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'dg_data_source': 'public', 'dg_data_context'..."
1,america_health_rankings,America Health Rankings,US,2021-12-30 19:32:54.441000+00:00,2025-07-22 10:45:22.624000+00:00,"{'dg_data_source': 'public', 'dg_data_context'..."
2,austin_311,,US,2017-05-11 18:44:42.673000+00:00,2025-07-22 10:43:48.820000+00:00,"{'dg_data_source': 'public', 'dg_data_context'..."


In [22]:
os.environ["description_parquet"] = parquet_file = "bigquery.dataset.metadata.description.parquet"
dataset_metadata_df.drop(columns=["Access Entries"]).to_parquet( parquet_file, index=False)


In [25]:
!ls -l {parquet_file} --si


-rw-r--r-- 1 root root 64k Nov 16 20:10 bigquery.dataset.metadata.description.parquet


In [20]:
duckdb.query(f"select count(1) from '{parquet_file}' ;")


┌──────────┐
│ count(1) │
│  int64   │
├──────────┤
│      382 │
└──────────┘

In [21]:
duckdb.query(f'''
  select *
  from '{parquet_file}'
  limit 10
  ;
''').df()


,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
1,america_health_rankings,America Health Rankings,US,2021-12-30 19:32:54.441000+00:00,2025-07-22 10:45:22.624000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
2,austin_311,,US,2017-05-11 18:44:42.673000+00:00,2025-07-22 10:43:48.820000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
3,austin_bikeshare,Austin Bikeshare dataset,US,2017-05-16 19:25:37.054000+00:00,2025-07-22 10:44:24.011000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
4,austin_crime,Austin Crime dataset,US,2017-05-11 18:29:25.469000+00:00,2025-07-22 10:45:15.608000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
5,austin_incidents,,US,2017-05-16 19:25:51.354000+00:00,2025-07-22 10:45:12.508000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
6,austin_waste,austin waste and diversion,US,2017-05-16 19:25:20.941000+00:00,2025-07-22 10:45:18.110000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
7,baseball,Data Updated On : 2016-10-24\nOverview: This p...,US,2016-10-18 17:08:24.118000+00:00,2025-07-23 09:17:37.113000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
8,bbc_news,Data Updated On : 2017-06-07\n,US,2017-06-06 15:21:18.540000+00:00,2026-10-13 07:12:32.569000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."
9,biglake-public-nyc-taxi-iceberg.public_data,None,us-central1,2025-11-14 17:50:35.493000+00:00,2025-11-14 17:50:35.493000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."


## Push description parquet to Hugging Face



In [30]:
%%capture hf_upload
%%bash

# echo \
hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  ./${description_parquet} \
  ./${description_parquet}


In [31]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/BigQuery/commit/6d8df5f55ce9998f8cddfec9bf14f62f94d567b2



### Verify with DuckDB


In [32]:
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{parquet_file}"
hf_url


'https://huggingface.co/datasets/rwcitek/BigQuery/resolve/main/bigquery.dataset.metadata.description.parquet'

In [33]:
duckdb.query(f'''
  select count(1) as 'number of data sets'
  from '{hf_url}'
  ;
''').df()


,number of data sets
0,382


In [34]:
duckdb.query(f'''
  select *
  from '{hf_url}'
  limit 1
;
''').df()


,Dataset ID,Description,Location,Created Time,Last Modified Time,Labels
0,agent_analytic_sample_dataset,Example dataset to demonstrate agent analytics...,US,2026-01-27 05:11:32.156000+00:00,2026-01-29 08:51:44.916000+00:00,"{'census_bureau': None, 'dataplex-data-documen..."


## Get number of rows, columns, and bytes

In [35]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

data = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
  # if i >= 3: break
  # if i >= 50: break
  # if i >= 50: break

  dataset_id = dataset.dataset_id
  print(f"{i:03}: {dataset_id}")
  tables = client.list_tables(dataset)  # Make an API request.
  try:
    for table in tables:
      table_ref = client.get_table(table)  # Make an API request.
      table_id = table_ref.table_id
      num_rows = table_ref.num_rows
      num_columns = len(table_ref.schema)
      size_mb = round(table_ref.num_bytes / (1000 ** 2),2)
      modified = table_ref.modified

      # Append the data to the list
      data.append({
        "Dataset ID": dataset_id,
        'Table': table_id,
        'Number of Rows': num_rows,
        'Number of Columns': num_columns,
        'Size in MB': size_mb,
        'Last Modified': modified,
    })
  except:
    data.append({
      "Dataset ID": dataset_id,
      'Table': None,
      'Number of Rows': 0,
      'Number of Columns': 0,
      'Size in MB': None,
      'Last Modified': None,
    })


# Create a DataFrame from the list of data
table_df = pd.DataFrame(data)
table_df[:10]

000: agent_analytic_sample_dataset
001: america_health_rankings
002: austin_311
003: austin_bikeshare
004: austin_crime
005: austin_incidents
006: austin_waste
007: baseball
008: bbc_news
009: biglake-public-nyc-taxi-iceberg.public_data
010: bigqueryml_ncaa
011: bitcoin_blockchain
012: blackhole_database
013: blockchain_analytics_ethereum_mainnet_us
014: bls
015: bls_qcew
016: breathe
017: broadstreet_adi
018: catalonian_mobile_coverage
019: catalonian_mobile_coverage_eu
020: census_bureau_acs
021: census_bureau_construction
022: census_bureau_international
023: census_bureau_usa
024: census_opportunity_atlas
025: census_utility
026: cfpb_complaints
027: chicago_crime
028: chicago_taxi_trips
029: clemson_dice
030: cloud_storage_geo_index
031: cms_codes
032: cms_medicare
033: cms_synthetic_patient_data_omop
034: country_codes
035: covid19_aha
036: covid19_covidtracking
037: covid19_ecdc
038: covid19_ecdc_eu
039: covid19_genome_sequence
040: covid19_geotab_mobility_impact
041: covid19_ge

,Dataset ID,Table,Number of Rows,Number of Columns,Size in MB,Last Modified
0,agent_analytic_sample_dataset,agent_events,16,8,0.00,2026-01-23 09:36:11.926000+00:00
1,agent_analytic_sample_dataset,agent_events_v2,36,16,0.02,2026-01-23 09:36:11.450000+00:00
2,agent_analytic_sample_dataset,session_embeddings,5,3,0.03,2026-01-23 09:36:11.259000+00:00
3,america_health_rankings,ahr,18155,10,2.96,2026-11-16 10:03:22.707000+00:00
4,america_health_rankings,america_health_rankings,0,0,0.00,2021-12-30 19:32:54.848000+00:00
5,austin_311,311_service_requests,2586511,22,712.61,2026-11-16 10:42:02.319000+00:00
6,austin_bikeshare,bikeshare_stations,101,16,0.02,2026-11-16 10:03:28.024000+00:00
7,austin_bikeshare,bikeshare_trips,2916659,10,332.67,2026-11-16 10:07:23.072000+00:00
8,austin_bikeshare,bikeshare_trips_daily_counts,3851,2,0.06,2025-10-27 21:26:21.264000+00:00
9,austin_bikeshare,daily_bikeshare_trips,3851,2,0.06,2026-02-23 17:12:03.841000+00:00


In [37]:
table_df.describe().transpose()


,count,mean,std,min,25%,50%,75%,max
Number of Rows,10_477.00,6_919_364_562.47,342_102_904_038.60,0.00,156.00,2_443.00,213_857.00,30_538_510_552_291.00
Number of Columns,10_477.00,44.30,119.37,0.00,8.00,17.00,41.00,7_897.00
Size in MB,10_451.00,1_063_865.51,36_581_252.86,0.00,0.04,1.47,97.89,2_604_707_130.70


In [ ]:


# stop



## Create parquet from data frame


In [38]:
os.environ["table_parquet"] = parquet_file = "bigquery.dataset.metadata.tables.parquet"
table_df.to_parquet( parquet_file, index=False)


In [39]:
!ls -l big* --si


-rw-r--r-- 1 root root  64k Nov 16 20:10 bigquery.dataset.metadata.description.parquet
-rw-r--r-- 1 root root 250k Nov 16 21:02 bigquery.dataset.metadata.tables.parquet


In [40]:
duckdb.query(f"select count(1) from '{parquet_file}' ;")


┌──────────┐
│ count(1) │
│  int64   │
├──────────┤
│    10477 │
└──────────┘

In [41]:
duckdb.query(f"select * from '{parquet_file}' limit 10;").df()


,Dataset ID,Table,Number of Rows,Number of Columns,Size in MB,Last Modified
0,agent_analytic_sample_dataset,agent_events,16,8,0.00,2026-01-23 09:36:11.926000+00:00
1,agent_analytic_sample_dataset,agent_events_v2,36,16,0.02,2026-01-23 09:36:11.450000+00:00
2,agent_analytic_sample_dataset,session_embeddings,5,3,0.03,2026-01-23 09:36:11.259000+00:00
3,america_health_rankings,ahr,18155,10,2.96,2026-11-16 10:03:22.707000+00:00
4,america_health_rankings,america_health_rankings,0,0,0.00,2021-12-30 19:32:54.848000+00:00
5,austin_311,311_service_requests,2586511,22,712.61,2026-11-16 10:42:02.319000+00:00
6,austin_bikeshare,bikeshare_stations,101,16,0.02,2026-11-16 10:03:28.024000+00:00
7,austin_bikeshare,bikeshare_trips,2916659,10,332.67,2026-11-16 10:07:23.072000+00:00
8,austin_bikeshare,bikeshare_trips_daily_counts,3851,2,0.06,2025-10-27 21:26:21.264000+00:00
9,austin_bikeshare,daily_bikeshare_trips,3851,2,0.06,2026-02-23 17:12:03.841000+00:00


## Push table parquet to Hugging Face



In [44]:
%%capture hf_upload
%%bash

# echo \
hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  ./${table_parquet} \
  ./${table_parquet}


In [45]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/BigQuery/commit/e13642ff7436bc6167377c0a6400e0205fbe924d



### Verify with DuckDB


In [46]:
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{parquet_file}"
hf_url


'https://huggingface.co/datasets/rwcitek/BigQuery/resolve/main/bigquery.dataset.metadata.tables.parquet'

In [47]:
duckdb.query(f'''
  select count(1) as 'number of tables'
  from '{hf_url}'
  ;
''').df()


,number of tables
0,10477


In [48]:
duckdb.query(f'''
  select *
  from '{hf_url}'
  limit 1
;
''').df()


,Dataset ID,Table,Number of Rows,Number of Columns,Size in MB,Last Modified
0,agent_analytic_sample_dataset,agent_events,16,8,0.00,2026-01-23 09:36:11.926000+00:00


## Some simple analytics with DuckDB


### Number of tables in a data set

#### Top 10 data sets with most tables

In [88]:
duckdb.query(f'''
  select "Dataset ID", count(1) as 'Number of Tables'
  from '{hf_url}'
  group by all
  order by "Number of Tables" DESC, "Dataset ID"
  limit 10
;
''').df()


,Dataset ID,Number of Tables
0,nlm_rxnorm,988
1,ebi_chembl,785
2,google_analytics_sample,369
3,census_bureau_acs,278
4,ghcn_d,268
5,idc_current_clinical,243
6,idc_v25_clinical,237
7,idc_v22_clinical,225
8,idc_v23_clinical,225
9,idc_v20_clinical,222


#### Top 10 data sets with fewest tables


In [89]:
duckdb.query(f'''
  select "Dataset ID", count(1) as 'Number of Tables'
  from '{hf_url}'
  group by all
  order by "Number of Tables", "Dataset ID"
  limit 10
;
''').df()


,Dataset ID,Number of Tables
0,austin_311,1
1,austin_crime,1
2,austin_waste,1
3,biglake-public-nyc-taxi-iceberg.public_data,1
4,bigqueryml_ncaa,1
5,blackhole_database,1
6,catalonian_mobile_coverage,1
7,catalonian_mobile_coverage_eu,1
8,census_bureau_construction,1
9,cfpb_complaints,1


### Size of data sets in bytes

#### Top 10 largest data sets


In [90]:
duckdb.query(f'''
  select "Dataset ID", sum("Size in MB")/1000/1000 as 'Size in TB'
  from '{hf_url}'
  group by all
  order by "Size in TB" DESC
  limit 10
;
''').df()


,Dataset ID,Size in TB
0,national_water_model,"4,091.37"
1,pypi,"2,792.13"
2,crypto_solana_mainnet_us,"2,082.25"
3,deps_dev_v1,617.73
4,google_books_ngrams_2020,381.09
5,noaa_global_forecast_system,272.03
6,noaa_gfs_025_degree_0th_hour_estimates_via_goo...,150.36
7,crypto_sui_mainnet_us,145.72
8,crypto_aptos_mainnet_us,111.63
9,crypto_aptos_testnet_us,93.91


#### Top 10 smallest data sets


In [91]:
duckdb.query(f'''
  select "Dataset ID", sum("Size in MB") as 'Size in MB'
  from '{hf_url}'
  group by all
  having "Size in MB" > 0
  order by "Size in MB"
  limit 10
;
''').df()


,Dataset ID,Size in MB
0,country_codes,0.01
1,ml_datasets_uscentral1,0.03
2,agent_analytic_sample_dataset,0.05
3,covid19_covidtracking,0.06
4,labeled_patents,0.11
5,world_bank_global_population,0.13
6,san_francisco_neighborhoods,0.17
7,google_dei,0.19
8,census_utility,0.20
9,us_res_real_est_data,0.40


### Size of tables ( in GB ) in each data set.

#### The Top 10 largest tables


In [92]:
duckdb.query(f'''
  select "Dataset ID", "Table", sum("Size in MB")/1000/1000 as 'Size in TB'
  from '{hf_url}'
  group by all
  order by "Size in TB" DESC
  limit 10
;
''').df()


,Dataset ID,Table,Size in TB
0,national_water_model,medium_range_channel_rt,"2,604.71"
1,pypi,file_downloads,"1,470.82"
2,pypi,simple_requests,"1,321.12"
3,crypto_solana_mainnet_us,Transactions,"1,018.75"
4,crypto_solana_mainnet_us,Instructions,"1,004.27"
5,national_water_model,long_range_channel_rt,938.04
6,deps_dev_v1,DependencyGraphEdges,353.56
7,national_water_model,short_range_channel_rt,290.97
8,noaa_global_forecast_system,NOAA_GFS0P25,272.03
9,noaa_gfs_025_degree_0th_hour_estimates_via_goo...,Observation,150.34


#### The Top 10 smallest tables


In [93]:
duckdb.query(f'''
  select "Dataset ID", "Table", sum("Size in MB") as 'Size in MB'
  from '{hf_url}'
  group by all
  having "Size in MB" >0
  order by "Size in MB"
  limit 10
;
''').df()


,Dataset ID,Table,Size in MB
0,crypto_tezos,double_endorsement_evidence_operations,0.01
1,ebi_chembl,activity_stds_lookup,0.01
2,ebi_chembl,activity_stds_lookup_24,0.01
3,census_utility,lsad_codes,0.01
4,ebi_chembl,activity_stds_lookup_25,0.01
5,cms_synthetic_patient_data_omop,concept_class,0.01
6,ebi_chembl,activity_stds_lookup_27,0.01
7,census_bureau_international,country_names_area,0.01
8,ebi_chembl,activity_stds_lookup_23,0.01
9,ebi_chembl,activity_stds_lookup_28,0.01


### Number of rows per table per data set



#### The Top 10 tables with the most rows


In [94]:
duckdb.query(f'''
  select "Dataset ID", "Table", "Number of Rows"//1000//1000//1000 as "Number of Rows (1e9)"
  from '{hf_url}'
  order by "Number of Rows (1e9)" DESC
  limit 10
;
''').df().style.format({'Number of Rows (1e9)': '{:,}'})


,Dataset ID,Table,Number of Rows (1e9)
0,national_water_model,medium_range_channel_rt,"30,538"
1,national_water_model,long_range_channel_rt,"14,656"
2,pypi,simple_requests,"6,142"
3,pypi,file_downloads,"3,736"
4,national_water_model,short_range_channel_rt,"3,306"
5,deps_dev_v1,DependencyGraphEdges,"2,804"
6,national_water_model,medium_range_no_da_channel_rt,"1,632"
7,deps_dev_v1,Dependencies,"1,297"
8,crypto_solana_mainnet_us,Instructions,"1,223"
9,deps_dev_v1,Dependents,"1,119"


#### The Top 10 tables with the fewest rows


In [95]:
duckdb.query(f'''
  select "Dataset ID", "Table", "Number of Rows"
  from '{hf_url}'
  where "Number of Rows" > 0
  order by "Number of Rows"
  limit 10
;
''').df()


,Dataset ID,Table,Number of Rows
0,ebi_chembl,version,1
1,ebi_chembl,version_23,1
2,ebi_chembl,version_24,1
3,ebi_chembl,version_25,1
4,ebi_chembl,version_26,1
5,ebi_chembl,version_27,1
6,ebi_chembl,version_28,1
7,ebi_chembl,version_29,1
8,ebi_chembl,version_33,1
9,geo_us_boundaries,national_outline,1
